# Sticky Regime Model — V1
## NVDA, META, GOOGL, TSLA, MSFT, AMZN, AAPL

---

## Objectif

Expliquer le mouvement du smile de volatilité entre **t0** et **t1** par une combinaison des trois régimes sticky :

| Régime | Définition |
|---|---|
| **Sticky Strike (SS)** | Les vols pour des strikes absolus $(K, T)$ ne bougent pas. La surface `vol_t1 = vol_t0` en strike absolu. |
| **Sticky Delta (SD)** | La surface en moneyness $(K/S, \tau)$ reste fixe. Quand le spot bouge, la surface se translate avec lui. |
| **Sticky Skew (SK)** | La vol ATM se déplace de $2 \times \text{skew} \times \Delta\ln S$ — le régime Local Vol. |

## Pourquoi l'OLS sur `dVol` était instable

Quand $\Delta S \approx 0$, les trois courbes `d_sticky` sont presque identiques → les régresseurs sont colinéaires → coefficients instables, R² aléatoire.

## Solution : projection sur les niveaux

Au lieu de régresser `dVol` sur les `d_sticky`, on projette **`vol_t1(K)`** directement sur les trois scénarios de **niveau** :

$$\min_{w_{SS}, w_{SD}, w_{SK}} \sum_K \text{Vega}(K) \cdot \left[ vol_{t1}(K) - \left( w_{SS} \cdot v^{SS}(K) + w_{SD} \cdot v^{SD}(K) + w_{SK} \cdot v^{SK}(K) \right) \right]^2$$

Contraintes : $w_{SS} + w_{SD} + w_{SK} = 1$, $w_i \geq 0$.

Les trois scénarios restent **distincts même quand $\Delta S \approx 0$** car ce sont des niveaux de vol et non des différences.

## SSR de Bergomi IV

En parallèle on calcule le **Skew Stickiness Ratio** :
$$\tilde{\beta} = \frac{\text{Cov}(\Delta\sigma_{ATM}, \Delta\ln S)}{\text{Var}(\Delta\ln S)} \div |\text{skew}_{ATM}|$$

- $\tilde{\beta} \approx 0$ → Sticky Delta
- $\tilde{\beta} \approx 1$ → Sticky Strike
- $\tilde{\beta} \approx 2$ → Sticky Skew

C'est une mesure indépendante qui valide la décomposition.


---
## Section 0 — Imports & Configuration

In [ ]:
import os
import warnings
import pickle

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize

warnings.simplefilter("ignore")

print("Imports OK")

In [ ]:
# ============================================================
#  CONNEXION MDX
# ============================================================
import ezmdx
from maxxpy.apis.mdx.api import MdxClient

LOGIN_MDX    = ""    # <-- TON LOGIN
PASSWORD_MDX = ""    # <-- TON MOT DE PASSE

ezmdx.set_app(app_name="VEGA5")
ezmdx.prod.satis_login()

mtx_client = MdxClient("MSD", LOGIN_MDX, PASSWORD_MDX, use_prod_only=True)
print("MDX OK")

In [ ]:
# ============================================================
#  PARAMÈTRES GLOBAUX — MODIFIER ICI
# ============================================================
from pandas.tseries.offsets import BDay

# ─── Dates ────────────────────────────────────────────────
# t1 = aujourd'hui (dernier jour ouvré), t0 = veille
t1 = pd.Timestamp.today().normalize() - BDay(1)
t0 = t1 - BDay(1)

DATE_T0 = t0.strftime("%Y-%m-%d")
DATE_T1 = t1.strftime("%Y-%m-%d")

print(f"t0 = {DATE_T0}")
print(f"t1 = {DATE_T1}")

# ─── Univers d'actifs ─────────────────────────────────────
ASSETS = [
    "NVDA",   # Nvidia
    "META",   # Meta / Facebook
    "GOOGL",  # Google
    "TSLA",   # Tesla
    "MSFT",   # Microsoft
    "AMZN",   # Amazon
    "AAPL",   # Apple
]

# ─── Filtres ──────────────────────────────────────────────
MONEYNESS_MIN = 0.80   # 80% du spot
MONEYNESS_MAX = 1.20   # 120% du spot
MIN_STRIKES   = 3      # minimum de strikes par slice pour continuer

# ─── Paramètres de taux ───────────────────────────────────
R = 0.0
Q = 0.0

# ─── Chemins ──────────────────────────────────────────────
CACHE_PATH = "./market_cache_v1.pkl"
EXCEL_PATH = "./sticky_results_v1.xlsx"

# MDX types
MDX_VOL_TYPE  = "EQUITY_VOLATILITY"
MDX_SPOT_TYPE = ["STOCK_QUOTE", "INDEX_QUOTE", "FUND_QUOTE"]

---
## Section 1 — Récupération des données MDX

In [ ]:
# ============================================================
#  FONCTIONS MDX
# ============================================================
def fetch_vols_mdx(ticker, dates, mtx_client, asset_type="S"):
    """
    Récupère les vols implicites pour un ticker US sur les dates données.
    asset_type = 'S' pour stock US.
    """
    code = f"{asset_type}_{ticker}"
    try:
        df = mtx_client.get_market_data(
            mdx_type=MDX_VOL_TYPE,
            code=code,
            date=dates
        )
        if df is None or len(df) == 0:
            return None
        return df[["STRIKE", "MATURITY", "VOLATILITY", "DATE"]].copy()
    except Exception as e:
        print(f"  Erreur vols {ticker}: {e}")
        return None


def fetch_spot_mdx(ticker, dates, mtx_client):
    """
    Récupère le spot pour un ticker US.
    Essaie plusieurs types MDX jusqu'à succès.
    """
    for mdx_type in MDX_SPOT_TYPE:
        try:
            df = mtx_client.get_market_data(
                mdx_type=mdx_type,
                code=ticker,
                date=dates
            )
            if df is None or len(df) == 0:
                continue
            df["DATE"] = pd.to_datetime(df["DATE"])
            price_col = [c for c in df.columns if c not in ["DATE"]][0]
            df["MID"] = pd.to_numeric(df[price_col], errors="coerce")
            return df[["DATE", "MID"]].dropna()
        except Exception:
            continue
    print(f"  Spot non disponible pour {ticker}")
    return None

In [ ]:
# ============================================================
#  CHARGEMENT DES DONNÉES (avec cache)
# ============================================================
def load_market_data(assets, date_t0, date_t1, mtx_client, cache_path, force_refresh=False):
    """
    Charge les données de marché pour tous les assets.
    Utilise un cache pickle pour éviter de re-fetcher.
    """
    if os.path.exists(cache_path) and not force_refresh:
        print(f"Cache trouvé : {cache_path}")
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        # Vérifier que les dates correspondent
        if (cache.get("meta", {}).get("date_t0") == date_t0
                and cache.get("meta", {}).get("date_t1") == date_t1):
            print(f"  Dates OK : t0={date_t0}, t1={date_t1}")
            return cache
        else:
            print("  Dates différentes → refresh")

    print("Fetching depuis MDX...")
    dates = [date_t0, date_t1]

    cache = {
        "meta": {
            "date_t0": date_t0,
            "date_t1": date_t1,
            "t0": pd.Timestamp(date_t0),
            "t1": pd.Timestamp(date_t1),
            "assets": assets,
        },
        "data": {}
    }

    for ticker in assets:
        print(f"  Fetching {ticker}...")
        vols = fetch_vols_mdx(ticker, dates, mtx_client)
        spot = fetch_spot_mdx(ticker, dates, mtx_client)

        cache["data"][ticker] = {"vols": vols, "spots": spot}

        if vols is not None:
            print(f"    Vols : {len(vols)} lignes")
        if spot is not None:
            print(f"    Spot : {len(spot)} lignes")

    with open(cache_path, "wb") as f:
        pickle.dump(cache, f)
    print(f"Cache sauvegardé : {cache_path}")
    return cache


# ─── Chargement ───────────────────────────────────────────
# Mettre force_refresh=True pour forcer un nouveau fetch
market_cache = load_market_data(
    ASSETS, DATE_T0, DATE_T1, mtx_client, CACHE_PATH, force_refresh=False
)

print(f"\nDonnées chargées pour : {list(market_cache['data'].keys())}")

---
## Section 2 — Fonctions utilitaires

### 2.1 Black-Scholes & interpolation
### 2.2 Construction des trois courbes sticky
### 2.3 Projection sur les niveaux (méthode principale)

In [ ]:
# ============================================================
#  UTILITAIRES BLACK-SCHOLES
# ============================================================
def bs_delta(S, K, T, sigma, r=0., q=0.):
    """Delta BS d'un call."""
    if T <= 0 or sigma <= 0:
        return 0.
    d1 = (np.log(S / K) + T * (r - q + 0.5 * sigma**2)) / (sigma * np.sqrt(T))
    return np.exp(-q * T) * norm.cdf(d1)


def bs_vega(S, K, T, sigma, r=0., q=0.):
    """Vega BS (sensibilité du prix à la vol)."""
    if T <= 0 or sigma <= 0:
        return 0.
    d1 = (np.log(S / K) + T * (r - q + 0.5 * sigma**2)) / (sigma * np.sqrt(T))
    return S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)


def bs_price(S, K, T, sigma, r=0., q=0., option="call"):
    """Prix BS d'un call ou put."""
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.) if option == "call" else max(K - S, 0.)
    d1 = (np.log(S / K) + T * (r - q + 0.5 * sigma**2)) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == "call":
        return (S * np.exp(-q * T) * norm.cdf(d1)
                - K * np.exp(-r * T) * norm.cdf(d2))
    return (K * np.exp(-r * T) * norm.cdf(-d2)
            - S * np.exp(-q * T) * norm.cdf(-d1))


from scipy.optimize import brentq

def implied_vol(S, K, T, price, r=0., q=0., option="call"):
    """Vol implicite BS par méthode de Brent."""
    try:
        return brentq(
            lambda v: bs_price(S, K, T, v, r, q, option) - price,
            1e-6, 5.0, xtol=1e-8
        )
    except Exception:
        return np.nan


def business_days(date_mat, date_ref):
    """Nombre de jours ouvrés entre date_ref et date_mat."""
    return max(len(pd.bdate_range(date_ref, date_mat)) - 1, 0)


print("Utilitaires BS OK")

In [ ]:
# ============================================================
#  INTERPOLATION LINÉAIRE SUR UNE GRILLE DE STRIKES
# ============================================================
def interp_vol(K_obs, vol_obs, K_target):
    """
    Interpole (extrapolation plate aux bords) la vol sur K_target.
    K_obs, vol_obs : arrays des observations
    K_target : array des strikes cibles
    """
    sort_idx = np.argsort(K_obs)
    K_s = K_obs[sort_idx]
    v_s = vol_obs[sort_idx]
    return np.interp(K_target, K_s, v_s)


print("Interpolation OK")

In [ ]:
# ============================================================
#  CONSTRUCTION DES TROIS COURBES STICKY
# ============================================================

def build_sticky_strike(K_grid, K_obs_t0, vol_obs_t0):
    """
    Sticky Strike : vol_t1(K) = vol_t0(K) pour les mêmes strikes absolus.
    La surface ne bouge pas du tout en strike absolu.
    """
    return interp_vol(K_obs_t0, vol_obs_t0, K_grid)


def build_sticky_delta(K_grid, K_obs_t0, vol_obs_t0, S0, S1, T0, T1, r=0., q=0.):
    """
    Sticky Delta : la surface en delta reste fixe.
    Pour chaque strike K à t1, on trouve le strike K' à t0 qui avait
    le même delta, et on prend vol_t0(K').

    Méthode :
    1. Calculer le delta BS de chaque K sur la grille à t1 avec vol_t0 interpolée
    2. Trouver le strike K' à t0 qui avait ce même delta
    3. vol_SD(K) = vol_t0(K')

    Approximation robuste : on mappe via le log-moneyness normalisé.
    Delta à t0 pour K' : Δ₀(K') = N(d1(S0, K', T0, σ0(K')))
    On cherche K' tel que Δ₀(K') = Δ₁(K) où Δ₁(K) = N(d1(S1, K, T1, σ1_approx(K)))

    Approche simplifiée (market convention) :
    K' = K * (S0 / S1)  — on mappe en ajustant le strike par le ratio de spot.
    C'est l'approximation standard utilisée en pratique.
    """
    # Strike équivalent à t0 pour le même delta : K' = K * S0/S1
    K_equiv = K_grid * (S0 / S1)
    return interp_vol(K_obs_t0, vol_obs_t0, K_equiv)


def build_sticky_skew(K_grid, K_obs_t0, vol_obs_t0, S0, S1, r=0., q=0.):
    """
    Sticky Skew (Local Vol-like) :
    La vol ATM se déplace de 2 × skew × ΔlnS.
    L'ensemble du smile se translate verticalement.

    vol_SK(K) = vol_t0(K) + [vol_t0(S1) - vol_t0(S0)]
                          + dσ/dK * (K - K)  ← terme de skew

    Formulation correcte (mouvement du smile avec le spot) :
    vol_SK(K) = vol_t0(K - ΔS) + [vol_t0(S1) - vol_t0(S0)]

    C'est l'approximation de la règle Sticky Skew :
    le smile se déplace avec le spot tout en maintenant son niveau ATM cohérent.
    """
    dS = S1 - S0

    # Vol ATM à t0 et à t1 selon t0
    sigma_t0_S0 = interp_vol(K_obs_t0, vol_obs_t0, np.array([S0]))[0]
    sigma_t0_S1 = interp_vol(K_obs_t0, vol_obs_t0, np.array([S1]))[0]

    # Le smile se translate : on lit vol_t0 au strike décalé de -ΔS
    # puis on ajoute le shift ATM
    vol_shifted = interp_vol(K_obs_t0, vol_obs_t0, K_grid - dS)
    atm_shift   = sigma_t0_S1 - sigma_t0_S0

    return vol_shifted - interp_vol(K_obs_t0, vol_obs_t0, K_grid - dS - dS) * 0 + atm_shift
    # Simplification propre :
    # vol_SK(K) = vol_t0(K - dS) + Δσ_ATM


def build_sticky_skew_v2(K_grid, K_obs_t0, vol_obs_t0, S0, S1):
    """
    Version propre et directe du Sticky Skew.
    vol_SK(K) = vol_t0(K - ΔS) + [vol_t0(S1) - vol_t0(S0)]
    """
    dS = S1 - S0
    sigma_t0_S0 = interp_vol(K_obs_t0, vol_obs_t0, np.array([S0]))[0]
    sigma_t0_S1 = interp_vol(K_obs_t0, vol_obs_t0, np.array([S1]))[0]
    delta_atm   = sigma_t0_S1 - sigma_t0_S0

    vol_translated = interp_vol(K_obs_t0, vol_obs_t0, K_grid - dS)
    return vol_translated + delta_atm


print("Fonctions sticky OK")

In [ ]:
# ============================================================
#  PROJECTION SUR LES NIVEAUX — MÉTHODE PRINCIPALE
# ============================================================

def project_on_sticky_levels(vol_t1, v_SS, v_SD, v_SK, weights):
    """
    Projette vol_t1 sur la combinaison convexe des trois scénarios sticky.

    Résout :
        min_{w_SS, w_SD, w_SK} Σ_K w(K) * [vol_t1(K) - (w_SS*v_SS(K) + w_SD*v_SD(K) + w_SK*v_SK(K))]²
    s.c. w_SS + w_SD + w_SK = 1
         w_i >= 0

    Paramètres
    ----------
    vol_t1  : array (n_strikes,) — vol observée à t1
    v_SS    : array (n_strikes,) — scénario Sticky Strike
    v_SD    : array (n_strikes,) — scénario Sticky Delta
    v_SK    : array (n_strikes,) — scénario Sticky Skew
    weights : array (n_strikes,) — poids (typiquement vega)

    Retourne
    --------
    w_SS, w_SD, w_SK : poids optimaux (somme = 1)
    vol_pred         : vol reconstruite
    r2               : R² de la projection
    r2_weighted      : R² pondéré
    """
    y = vol_t1
    X = np.column_stack([v_SS, v_SD, v_SK])
    w = weights / (weights.mean() + 1e-12)

    # Fonction objectif pondérée
    def objective(params):
        wss, wsd, wsk = params
        y_pred = wss * v_SS + wsd * v_SD + wsk * v_SK
        return np.sum(w * (y - y_pred)**2)

    constraints = [
        {"type": "eq",  "fun": lambda p: p[0] + p[1] + p[2] - 1.0},
    ]
    bounds = [(0., 1.), (0., 1.), (0., 1.)]

    # Plusieurs points de départ pour éviter les minima locaux
    starting_points = [
        [1/3, 1/3, 1/3],
        [0.6, 0.2, 0.2],
        [0.2, 0.6, 0.2],
        [0.2, 0.2, 0.6],
        [0.8, 0.1, 0.1],
        [0.1, 0.8, 0.1],
        [0.1, 0.1, 0.8],
        [0.5, 0.5, 0.0],
        [0.5, 0.0, 0.5],
        [0.0, 0.5, 0.5],
    ]

    best_obj = np.inf
    best_x   = np.array([1/3, 1/3, 1/3])

    for x0 in starting_points:
        try:
            res = minimize(
                objective,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 2000, "ftol": 1e-14}
            )
            if res.success and objective(res.x) < best_obj:
                best_obj = objective(res.x)
                best_x   = res.x
        except Exception:
            continue

    w_SS, w_SD, w_SK = best_x
    vol_pred = w_SS * v_SS + w_SD * v_SD + w_SK * v_SK

    # R² classique (sur les niveaux)
    ss_res = np.sum((y - vol_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-14 else np.nan

    # R² pondéré
    y_bar_w   = np.sum(w * y) / np.sum(w)
    ss_res_w  = np.sum(w * (y - vol_pred)**2)
    ss_tot_w  = np.sum(w * (y - y_bar_w)**2)
    r2_w = 1 - ss_res_w / ss_tot_w if ss_tot_w > 1e-14 else np.nan

    # R² sur les DIFFÉRENCES de vol (mesure complémentaire)
    dVol_obs  = y - v_SS  # dVol par rapport à sticky strike
    dVol_pred = vol_pred - v_SS
    ss_res_d = np.sum((dVol_obs - dVol_pred)**2)
    ss_tot_d = np.sum((dVol_obs - dVol_obs.mean())**2)
    r2_diff = 1 - ss_res_d / ss_tot_d if ss_tot_d > 1e-14 else np.nan

    return w_SS, w_SD, w_SK, vol_pred, r2, r2_w, r2_diff


print("Projection OK")

In [ ]:
# ============================================================
#  SSR DE BERGOMI IV
# ============================================================

def compute_SSR(vol_t0_atm, vol_t1_atm, dS, S0, skew_atm_t0):
    """
    Skew Stickiness Ratio de Bergomi IV.

    SSR = (Δσ_ATM / ΔlnS) / skew_ATM_t0

    Interprétation :
      SSR ≈ 0 → Sticky Delta (vol ATM ne bouge pas avec le spot)
      SSR ≈ 1 → Sticky Strike
      SSR ≈ 2 → Sticky Skew (Local Vol)

    Note : c'est une mesure instantanée (une seule date).
    Elle est bruitée mais donne une indication du régime du jour.
    """
    if abs(dS) < 1e-6 or abs(skew_atm_t0) < 1e-8 or S0 <= 0:
        return np.nan

    delta_sigma_atm = vol_t1_atm - vol_t0_atm
    delta_ln_S      = np.log((S0 + dS) / S0)

    if abs(delta_ln_S) < 1e-8:
        return np.nan

    ssr = (delta_sigma_atm / delta_ln_S) / skew_atm_t0
    return float(ssr)


def compute_skew_atm(K_obs, vol_obs, S, h=0.05):
    """
    Calcule le skew ATM : dσ/dlnK évalué en K=S.
    Approximation par différences finies.
    h : demi-largeur en fraction du spot (ex: 0.05 = 5%)
    """
    K_up = S * (1 + h)
    K_dn = S * (1 - h)
    vol_up = interp_vol(K_obs, vol_obs, np.array([K_up]))[0]
    vol_dn = interp_vol(K_obs, vol_obs, np.array([K_dn]))[0]
    # dσ/dlnK ≈ (vol_up - vol_dn) / (ln(K_up) - ln(K_dn))
    return (vol_up - vol_dn) / (np.log(K_up) - np.log(K_dn))


print("SSR Bergomi OK")

---
## Section 3 — Calcul principal

Pour chaque actif et chaque maturité disponible :
1. Récupérer les strikes de marché dans [80%, 120%] du spot
2. Construire les trois courbes sticky sur ces strikes
3. Projeter vol_t1 sur les niveaux → poids (w_SS, w_SD, w_SK)
4. Calculer les métriques complémentaires (R², SSR, erreurs)

In [ ]:
# ============================================================
#  BOUCLE PRINCIPALE
# ============================================================
all_rows = []

for asset in ASSETS:
    print(f"\n{'='*55}")
    print(f"Traitement : {asset}")
    print(f"{'='*55}")

    try:
        asset_data = market_cache["data"].get(asset, {})
        vols_raw   = asset_data.get("vols")  # DataFrame brut MDX
        spots_raw  = asset_data.get("spots") # DataFrame spot

        if vols_raw is None or spots_raw is None:
            print(f"  Données manquantes pour {asset}")
            continue

        # ── Spot ──────────────────────────────────────────────
        spots = spots_raw.copy()
        spots["DATE"] = pd.to_datetime(spots["DATE"])
        spots = spots.set_index("DATE")["MID"].astype(float)

        date_t0_ts = pd.Timestamp(DATE_T0)
        date_t1_ts = pd.Timestamp(DATE_T1)

        # Chercher le spot le plus proche si date exacte manquante
        S0 = float(spots.asof(date_t0_ts))
        S1 = float(spots.asof(date_t1_ts))

        if np.isnan(S0) or np.isnan(S1) or S0 <= 0 or S1 <= 0:
            print(f"  Spot invalide : S0={S0}, S1={S1}")
            continue

        dS     = S1 - S0
        dlnS   = np.log(S1 / S0)
        dS_pct = dlnS * 100

        print(f"  S0 = {S0:.2f}  |  S1 = {S1:.2f}  |  ΔlnS = {dS_pct:.3f}%")

        # ── Vols brutes ──────────────────────────────────────
        vols = vols_raw.copy()
        vols["DATE"]     = pd.to_datetime(vols["DATE"])
        vols["MATURITY"] = pd.to_datetime(vols["MATURITY"])
        vols["STRIKE"]   = pd.to_numeric(vols["STRIKE"],   errors="coerce")
        vols["vol"]      = pd.to_numeric(vols["VOLATILITY"], errors="coerce")

        # Les strikes MDX sont en % du spot → convertir en niveau absolu
        # STRIKE MDX = 100 * K/S  (convention MDX)
        spot_map = {date_t0_ts: S0, date_t1_ts: S1}
        vols["S_date"] = vols["DATE"].map(spot_map)
        vols["K"]      = vols["STRIKE"] * vols["S_date"] / 100.0

        # Convertir les vols en décimal si elles sont en pourcentage
        if vols["vol"].median() > 2:
            vols["vol"] = vols["vol"] / 100.0

        vols = vols.dropna(subset=["K", "vol", "DATE", "MATURITY"])

        # Séparer t0 et t1
        vols_t0 = vols[vols["DATE"] == date_t0_ts].copy()
        vols_t1 = vols[vols["DATE"] == date_t1_ts].copy()

        if vols_t0.empty or vols_t1.empty:
            print(f"  Vols vides pour {asset}")
            continue

        # ── Maturités communes ───────────────────────────────
        mats_t0 = set(vols_t0["MATURITY"].unique())
        mats_t1 = set(vols_t1["MATURITY"].unique())
        common_mats = sorted(mats_t0.intersection(mats_t1))

        print(f"  {len(common_mats)} maturités communes")

        # ── Boucle par maturité ──────────────────────────────
        for mat in common_mats:
            try:
                slice_t0 = vols_t0[vols_t0["MATURITY"] == mat].copy()
                slice_t1 = vols_t1[vols_t1["MATURITY"] == mat].copy()

                # Nombre de jours ouvrés
                T0_days = business_days(mat, date_t0_ts)
                T1_days = business_days(mat, date_t1_ts)

                if T0_days <= 0 or T1_days <= 0:
                    continue

                T0_years = T0_days / 252.0
                T1_years = T1_days / 252.0

                # Arrays strikes / vols
                K0_obs   = slice_t0["K"].to_numpy(dtype=float)
                vol0_obs = slice_t0["vol"].to_numpy(dtype=float)
                K1_obs   = slice_t1["K"].to_numpy(dtype=float)
                vol1_obs = slice_t1["vol"].to_numpy(dtype=float)

                if len(K0_obs) < MIN_STRIKES or len(K1_obs) < MIN_STRIKES:
                    continue

                # ── Grille de strikes : union des strikes de marché ──
                # Filtrer dans [80%, 120%] du spot t1 (référence)
                K_min_filter = S1 * MONEYNESS_MIN
                K_max_filter = S1 * MONEYNESS_MAX

                # Union des strikes t0 et t1 dans la zone de moneyness
                K_union = np.unique(np.round(
                    np.concatenate([K0_obs, K1_obs]), 2
                ))
                K_grid = K_union[
                    (K_union >= K_min_filter) & (K_union <= K_max_filter)
                ]

                if len(K_grid) < MIN_STRIKES:
                    continue

                # ── Interpolation des vols sur la grille ────────────
                vol0_grid = interp_vol(K0_obs, vol0_obs, K_grid)
                vol1_grid = interp_vol(K1_obs, vol1_obs, K_grid)

                # ── Trois scénarios sticky ──────────────────────────
                v_SS = build_sticky_strike(
                    K_grid, K0_obs, vol0_obs
                )
                v_SD = build_sticky_delta(
                    K_grid, K0_obs, vol0_obs, S0, S1, T0_years, T1_years, R, Q
                )
                v_SK = build_sticky_skew_v2(
                    K_grid, K0_obs, vol0_obs, S0, S1
                )

                # Vérification que les scénarios sont valides
                if (np.any(np.isnan(v_SS)) or np.any(np.isnan(v_SD))
                        or np.any(np.isnan(v_SK)) or np.any(np.isnan(vol1_grid))):
                    continue

                # ── Vega pour la pondération ─────────────────────────
                vega = np.array([
                    bs_vega(S0, K, T0_years, max(v, 1e-4), R, Q)
                    for K, v in zip(K_grid, vol0_grid)
                ])
                vega = np.clip(vega, 1e-10, None)

                # ── PROJECTION SUR LES NIVEAUX ──────────────────────
                w_SS, w_SD, w_SK, vol_pred, r2, r2_w, r2_diff = project_on_sticky_levels(
                    vol1_grid, v_SS, v_SD, v_SK, vega
                )

                # ── SSR de Bergomi IV ────────────────────────────────
                # Vol ATM (interpolée au spot)
                vol_atm_t0 = interp_vol(K0_obs, vol0_obs, np.array([S0]))[0]
                vol_atm_t1 = interp_vol(K1_obs, vol1_obs, np.array([S1]))[0]

                skew_atm_t0 = compute_skew_atm(K0_obs, vol0_obs, S0)

                ssr = compute_SSR(vol_atm_t0, vol_atm_t1, dS, S0, skew_atm_t0)

                # Interprétation SSR
                if not np.isnan(ssr):
                    if ssr < 0.33:
                        ssr_regime = "Sticky Delta"
                    elif ssr < 1.33:
                        ssr_regime = "Sticky Strike"
                    elif ssr < 2.5:
                        ssr_regime = "Sticky Skew"
                    else:
                        ssr_regime = "Beyond Sticky Skew"
                else:
                    ssr_regime = "N/A (ΔS≈0)"

                # ── Part du mouvement expliquée par le spot ──────────
                # Mesure combien de la variation de vol ATM est due au spot
                # via la réponse sticky skew attendue
                expected_atm_move_SK = 2 * skew_atm_t0 * dlnS  # réponse SS complète
                actual_atm_move      = vol_atm_t1 - vol_atm_t0

                # Part du mouvement ATM expliquée par ΔS (via skew)
                if abs(expected_atm_move_SK) > 1e-8:
                    pct_spot_driven = min(abs(actual_atm_move / expected_atm_move_SK) * 100, 200)
                else:
                    pct_spot_driven = np.nan

                # ── Erreurs par scénario ─────────────────────────────
                rmse_SS = np.sqrt(np.mean((vol1_grid - v_SS)**2)) * 100
                rmse_SD = np.sqrt(np.mean((vol1_grid - v_SD)**2)) * 100
                rmse_SK = np.sqrt(np.mean((vol1_grid - v_SK)**2)) * 100
                rmse_pred = np.sqrt(np.mean((vol1_grid - vol_pred)**2)) * 100

                # ── Régime dominant ──────────────────────────────────
                weights_dict = {"Sticky Strike": w_SS, "Sticky Delta": w_SD, "Sticky Skew": w_SK}
                dominant_regime = max(weights_dict, key=weights_dict.get)

                print(f"  Maturité {mat.date()} (T={T1_days}j) : "
                      f"SS={w_SS*100:.1f}% SD={w_SD*100:.1f}% SK={w_SK*100:.1f}% "
                      f"| R²={r2:.3f} | SSR={ssr:.2f if not np.isnan(ssr) else 'N/A'}")

                # ── Construction du DataFrame de sortie ─────────────
                # Une ligne par strike
                for idx_k, K in enumerate(K_grid):
                    row = {
                        # ─ Identifiants ─
                        "UDL":            asset,
                        "Maturite":       mat,
                        "Date_t0":        date_t0_ts,
                        "Date_t1":        date_t1_ts,
                        "T_jours":        T1_days,
                        "T_annees":       round(T1_years, 4),

                        # ─ Spot ─
                        "Spot_t0":        round(S0, 4),
                        "Spot_t1":        round(S1, 4),
                        "dSpot":          round(dS, 4),
                        "dLnS_pct":       round(dS_pct, 4),

                        # ─ Strike ─
                        "Strike":         round(K, 4),
                        "Moneyness_pct":  round(K / S1 * 100, 4),

                        # ─ Vols observées ─
                        "Vol_t0":         round(vol0_grid[idx_k] * 100, 4),
                        "Vol_t1":         round(vol1_grid[idx_k] * 100, 4),
                        "dVol":           round((vol1_grid[idx_k] - vol0_grid[idx_k]) * 100, 4),

                        # ─ Trois scénarios sticky (niveaux) ─
                        "Vol_StickyStrike": round(v_SS[idx_k] * 100, 4),
                        "Vol_StickyDelta":  round(v_SD[idx_k] * 100, 4),
                        "Vol_StickySkew":   round(v_SK[idx_k] * 100, 4),

                        # ─ Différences sticky vs t0 ─
                        "dVol_StickyStrike": round((v_SS[idx_k] - vol0_grid[idx_k]) * 100, 4),
                        "dVol_StickyDelta":  round((v_SD[idx_k] - vol0_grid[idx_k]) * 100, 4),
                        "dVol_StickySkew":   round((v_SK[idx_k] - vol0_grid[idx_k]) * 100, 4),

                        # ─ Vol reconstruite ─
                        "Vol_Pred":         round(vol_pred[idx_k] * 100, 4),
                        "dVol_Pred":        round((vol_pred[idx_k] - vol0_grid[idx_k]) * 100, 4),
                        "Erreur_Pred":      round((vol1_grid[idx_k] - vol_pred[idx_k]) * 100, 4),

                        # ─ Poids des régimes ─
                        "Pct_StickyStrike": round(w_SS * 100, 2),
                        "Pct_StickyDelta":  round(w_SD * 100, 2),
                        "Pct_StickySkew":   round(w_SK * 100, 2),
                        "Regime_Dominant":  dominant_regime,

                        # ─ Métriques de qualité ─
                        "R2_niveaux":       round(r2, 4),
                        "R2_pondere":       round(r2_w, 4),
                        "R2_differences":   round(r2_diff, 4),
                        "RMSE_StickyStrike_vpts": round(rmse_SS, 4),
                        "RMSE_StickyDelta_vpts":  round(rmse_SD, 4),
                        "RMSE_StickySkew_vpts":   round(rmse_SK, 4),
                        "RMSE_Pred_vpts":          round(rmse_pred, 4),

                        # ─ SSR de Bergomi IV ─
                        "SSR_Bergomi":      round(ssr, 4) if not np.isnan(ssr) else np.nan,
                        "SSR_Regime":       ssr_regime,

                        # ─ Vol ATM ─
                        "Vol_ATM_t0":       round(vol_atm_t0 * 100, 4),
                        "Vol_ATM_t1":       round(vol_atm_t1 * 100, 4),
                        "dVol_ATM":         round((vol_atm_t1 - vol_atm_t0) * 100, 4),
                        "Skew_ATM_t0":      round(skew_atm_t0 * 100, 4),

                        # ─ Part spot ─
                        "Pct_SpotDriven":   round(pct_spot_driven, 2) if not np.isnan(pct_spot_driven) else np.nan,

                        # ─ Vega ─
                        "Vega":             round(vega[idx_k], 6),

                        # ─ Flags ─
                        "IsMarket_t0":      K in np.round(K0_obs, 2),
                        "IsMarket_t1":      K in np.round(K1_obs, 2),
                    }
                    all_rows.append(row)

            except Exception as e:
                print(f"  Erreur maturité {mat}: {e}")
                continue

    except Exception as e:
        print(f"Erreur {asset}: {e}")
        continue

print(f"\n{'='*55}")
print(f"Total lignes calculées : {len(all_rows)}")

---
## Section 4 — Résultats : vue synthétique par (UDL, Maturité)

In [ ]:
# ============================================================
#  DATAFRAME FINAL
# ============================================================
if len(all_rows) == 0:
    print("Aucun résultat. Vérifier les données MDX.")
else:
    df_final = pd.DataFrame(all_rows)
    df_final = df_final.sort_values(["UDL", "Maturite", "Strike"]).reset_index(drop=True)

    print(f"DataFrame final : {df_final.shape[0]} lignes × {df_final.shape[1]} colonnes")
    print(f"Colonnes : {list(df_final.columns)}")

In [ ]:
# ============================================================
#  VUE SYNTHÉTIQUE PAR (UDL, Maturité) — une ligne par slice
# ============================================================
if len(all_rows) > 0:
    df_synth = (
        df_final
        .drop_duplicates(subset=["UDL", "Maturite"])
        .sort_values(["UDL", "Maturite"])
        [["UDL", "Maturite", "Date_t0", "Date_t1",
          "Spot_t0", "Spot_t1", "dLnS_pct",
          "T_jours",
          "Vol_ATM_t0", "Vol_ATM_t1", "dVol_ATM",
          "Pct_StickyStrike", "Pct_StickyDelta", "Pct_StickySkew",
          "Regime_Dominant",
          "R2_niveaux", "R2_pondere", "R2_differences",
          "SSR_Bergomi", "SSR_Regime",
          "RMSE_Pred_vpts",
          "Pct_SpotDriven"]]
        .reset_index(drop=True)
    )

    print("\n" + "="*80)
    print("VUE SYNTHÉTIQUE — Poids des régimes sticky par (UDL, Maturité)")
    print("="*80)
    display(df_synth)

In [ ]:
# ============================================================
#  ANALYSE DE LA QUALITÉ DU MODÈLE
# ============================================================
if len(all_rows) > 0:
    print("\n" + "="*60)
    print("QUALITÉ DU MODÈLE")
    print("="*60)

    # Distribution du R² par méthode
    for col, label in [
        ("R2_niveaux",    "R² sur niveaux (méthode principale)"),
        ("R2_pondere",    "R² pondéré vega"),
        ("R2_differences", "R² sur différences de vol"),
    ]:
        r2_vals = df_synth[col].dropna()
        print(f"\n  {label}")
        print(f"    Médiane : {r2_vals.median():.4f}")
        print(f"    Moyenne : {r2_vals.mean():.4f}")
        print(f"    Min     : {r2_vals.min():.4f}")
        print(f"    Max     : {r2_vals.max():.4f}")

    print("\n  SSR de Bergomi IV (par UDL)")
    ssr_by_udl = df_synth.groupby("UDL")["SSR_Bergomi"].agg(["mean", "min", "max"])
    for udl, row in ssr_by_udl.iterrows():
        print(f"    {udl:<8} : SSR moyen = {row['mean']:.3f}  "
              f"[{row['min']:.3f}, {row['max']:.3f}]")

    print("\n  Régime dominant par UDL")
    regime_counts = df_synth.groupby(["UDL", "Regime_Dominant"]).size().unstack(fill_value=0)
    display(regime_counts)

---
## Section 5 — Export Excel

In [ ]:
# ============================================================
#  EXPORT EXCEL AVEC XLWINGS
# ============================================================
import xlwings as xw

if len(all_rows) == 0:
    print("Rien à exporter.")
else:
    # ── Ordre et nommage des colonnes pour Excel ────────────
    EXCEL_COLUMNS_ORDER = [
        "UDL",
        "Maturite",
        "Date_t0",
        "Date_t1",
        "T_jours",
        "T_annees",
        "Spot_t0",
        "Spot_t1",
        "dSpot",
        "dLnS_pct",
        "Strike",
        "Moneyness_pct",
        "Vol_t0",
        "Vol_t1",
        "dVol",
        "Vol_StickyStrike",
        "Vol_StickyDelta",
        "Vol_StickySkew",
        "dVol_StickyStrike",
        "dVol_StickyDelta",
        "dVol_StickySkew",
        "Vol_Pred",
        "dVol_Pred",
        "Erreur_Pred",
        "Pct_StickyStrike",
        "Pct_StickyDelta",
        "Pct_StickySkew",
        "Regime_Dominant",
        "R2_niveaux",
        "R2_pondere",
        "R2_differences",
        "RMSE_StickyStrike_vpts",
        "RMSE_StickyDelta_vpts",
        "RMSE_StickySkew_vpts",
        "RMSE_Pred_vpts",
        "SSR_Bergomi",
        "SSR_Regime",
        "Vol_ATM_t0",
        "Vol_ATM_t1",
        "dVol_ATM",
        "Skew_ATM_t0",
        "Pct_SpotDriven",
        "Vega",
        "IsMarket_t0",
        "IsMarket_t1",
    ]

    # Ne garder que les colonnes existantes dans le bon ordre
    cols_present = [c for c in EXCEL_COLUMNS_ORDER if c in df_final.columns]
    df_export = df_final[cols_present].copy()

    # ── Écriture Excel ────────────────────────────────────
    app = xw.App(visible=False)
    try:
        if os.path.exists(EXCEL_PATH):
            wb = app.books.open(EXCEL_PATH)
        else:
            wb = app.books.add()
            wb.save(EXCEL_PATH)

        # ── Feuille DATA — toutes les lignes ──
        try:
            sht_data = wb.sheets["DATA"]
        except Exception:
            sht_data = wb.sheets.add("DATA", after=wb.sheets[-1])
        sht_data.clear_contents()
        sht_data["A1"].value = df_export

        # ── Feuille SYNTHESE — une ligne par (UDL, Maturité) ──
        try:
            sht_synth = wb.sheets["SYNTHESE"]
        except Exception:
            sht_synth = wb.sheets.add("SYNTHESE", after=wb.sheets[-1])
        sht_synth.clear_contents()
        sht_synth["A1"].value = df_synth

        wb.save()
        wb.close()
        print(f"Excel exporté : {EXCEL_PATH}")
        print(f"  Feuille DATA     : {len(df_export)} lignes")
        print(f"  Feuille SYNTHESE : {len(df_synth)} lignes")

    except Exception as e:
        print(f"Erreur export Excel : {e}")
    finally:
        app.quit()

In [ ]:
# ============================================================
#  APERÇU FINAL
# ============================================================
if len(all_rows) > 0:
    print("\n" + "="*80)
    print("APERÇU — 20 premières lignes (DATA)")
    print("="*80)
    display(
        df_export[[
            "UDL", "Maturite", "Strike", "Moneyness_pct",
            "Vol_t0", "Vol_t1", "dVol",
            "Vol_StickyStrike", "Vol_StickyDelta", "Vol_StickySkew",
            "Pct_StickyStrike", "Pct_StickyDelta", "Pct_StickySkew",
            "R2_niveaux", "SSR_Bergomi", "SSR_Regime"
        ]].head(20)
    )

---

## Lecture des résultats

### Colonnes clés

| Colonne | Description |
|---|---|
| `Pct_StickyStrike` | % du mouvement du smile expliqué par Sticky Strike |
| `Pct_StickyDelta` | % expliqué par Sticky Delta |
| `Pct_StickySkew` | % expliqué par Sticky Skew |
| `Regime_Dominant` | Le régime avec le poids le plus élevé |
| `R2_niveaux` | R² de la projection sur les niveaux (robuste à ΔS faible) |
| `R2_differences` | R² sur les variations de vol (comparable à l'ancien OLS) |
| `SSR_Bergomi` | Skew Stickiness Ratio : 0=SD, 1=SS, 2=SK |
| `Pct_SpotDriven` | % du mouvement ATM expliqué par le spot |
| `RMSE_Pred_vpts` | Erreur de reconstruction en points de vol |

### Interprétation du SSR

- **SSR ≈ 0** : le marché est en Sticky Delta — la vol ne répond pas au spot
- **SSR ≈ 1** : Sticky Strike — les vols aux strikes fixes restent constantes
- **SSR ≈ 2** : Sticky Skew / Local Vol — réponse maximale de la vol ATM au spot
- **SSR hors [0,2]** : mouvement de vol autonome non lié au spot (choc idiosyncratique)

### Pourquoi le R² sur les niveaux est meilleur

L'OLS sur `dVol` échouait car les régresseurs `d_sticky` s'effondrent quand ΔS ≈ 0.
La projection sur les **niveaux** `vol_t1` reste bien conditionnée car les trois scénarios 
sticky sont distincts même sans mouvement de spot — ils diffèrent par la structure
de la surface entière et pas seulement par le shift ATM.